# Glucast — Detección anticipada de hipoglucemias
## TFM — Máster Universitario en Inteligencia Artificial (UNIR) · Pablo Daniel Figueroa

El RMSE mide el error **promedio**. Lo clínicamente crítico, en cambio, es otra cosa: **avisar a tiempo de una hipoglucemia sin generar tantas falsas alarmas que el paciente deje de prestarles atención**. Este notebook evalúa los modelos con ese criterio.

**Qué hace**
1. Reconstruye los datos y carga los modelos del experimento anterior (carpeta `resultados_…` en Drive). **No reentrena los modelos base** ni necesita el token de Nightscout.
2. Define los **eventos de hipoglucemia** según el consenso internacional (Battelino et al., 2019/2023): nivel 1 = glucosa < 70 mg/dL durante ≥ 15 min (3 lecturas seguidas); nivel 2 = < 54 mg/dL.
3. Compara **estrategias de alarma**:
   - **Umbral actual**: alarma si la glucosa *actual* es menor que θ. Es la alerta clásica del sensor; no anticipa.
   - **Tendencia lineal**: extrapola la pendiente de los últimos 15 min, como las alertas predictivas de los CGM.
   - **LSTM unificado (predicción puntual)**: alarma si la predicción es menor que θ.
   - **LSTM unificado + límite conformal inferior**: alarma si el *borde inferior* del intervalo de predicción baja de 70. Prioriza la seguridad.
   - **LSTM asimétrico**: el modelo unificado se reajusta con una **pérdida que penaliza mucho más sobreestimar la glucosa cuando está baja**, que es el error peligroso.
4. Mide **sensibilidad** (qué % de hipos se avisaron), **anticipación** (cuántos minutos antes) y **falsas alarmas por día**.
5. **Anti-sobreajuste:** los umbrales se eligen **solo con el conjunto de validación**. El conjunto de prueba se usa una única vez, al final, para informar el resultado.
6. Opcional: valida además sobre los **pacientes de prueba de OhioT1DM**, que tienen muchas más hipoglucemias. Así se comprueba que el resultado no depende de pocos eventos.

> GPU recomendada (*Entorno de ejecución → Cambiar tipo → T4*). Tiempo estimado: 5–15 min.

## 1. Librerías y configuración

In [ ]:
import os, glob, json, time, zipfile, getpass, warnings
import xml.etree.ElementTree as ET
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model, clone_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
warnings.filterwarnings('ignore'); tf.get_logger().setLevel('ERROR')
SEMILLA = 42; np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)
plt.rcParams['figure.figsize'] = (13, 4.5); plt.rcParams['font.size'] = 11

CARPETA_DRIVE = '/content/drive/MyDrive/MAESTRIA EN IA/TFM/ENTRENAMIENTO'
CARPETA_RES   = None        # None = usa la carpeta resultados_… más reciente
HORIZONTES    = [6, 12]     # 30 y 60 min
USAR_OHIO     = True        # validar también con pacientes de prueba de OhioT1DM (pide la contraseña)

# --- Mismos parámetros que el experimento anterior (no cambiar) ---
ZONA_HORARIA = 'America/Argentina/Buenos_Aires'
VENTANA = 24; SPLIT = (0.70, 0.15, 0.15)
FEATURES = ['glucosa', 'tendencia', 'bolo_u', 'iob', 'carbs', 'cob', 'hora_sin', 'hora_cos']
ESCALAS  = {'glucosa': 400, 'tendencia': 50, 'bolo_u': 10, 'iob': 10, 'carbs': 100, 'cob': 100, 'hora_sin': 1, 'hora_cos': 1}
ESCALA_DELTA = 100; DIA_PERIODOS = 24; ABSORCION_CARBS = 36

# --- Definiciones clínicas ---
HIPO = 70; HIPO_N2 = 54; MIN_LECTURAS_EVENTO = 3      # < 70 mg/dL durante ≥ 15 min
VENTANA_DETECCION_MIN = 60    # una alarma cuenta si ocurre hasta 60 min antes del inicio del evento
REFRACTARIO_MIN = 30          # una alarma que sigue activa vuelve a avisar cada 30 min
SENSIBILIDAD_OBJETIVO = 0.90  # al elegir umbrales: la mínima tasa de falsas alarmas que logre ≥ 90 % en validación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if CARPETA_RES is None:
    CARPETA_RES = sorted(d for d in glob.glob(os.path.join(CARPETA_DRIVE, 'resultados_*'))
                         if glob.glob(os.path.join(d, 'modelo_lstm_dataset_unificado_*.keras')))[-1]
CARPETA_HIPO = os.path.join(CARPETA_RES, 'deteccion_hipoglucemias'); os.makedirs(CARPETA_HIPO, exist_ok=True)
print('Usando modelos y datos de:', CARPETA_RES)

## 2. Reconstruir los datos (mismo preprocesamiento que el experimento)

In [ ]:
def construir_base(gluc, bolos, carbs):
    g = gluc.groupby(level=0).mean().resample('5min').mean()
    g = g.interpolate(method='linear', limit=3, limit_area='inside')
    df = pd.DataFrame({'glucosa': g})
    df['bolo_u'] = bolos.resample('5min').sum().reindex(df.index).fillna(0) if len(bolos) else 0.0
    df['carbs']  = carbs.resample('5min').sum().reindex(df.index).fillna(0) if len(carbs) else 0.0
    pesos_iob = np.maximum(0, 1 - np.arange(DIA_PERIODOS) / DIA_PERIODOS)
    pesos_cob = np.maximum(0, 1 - np.arange(ABSORCION_CARBS) / ABSORCION_CARBS)
    df['iob'] = np.convolve(df['bolo_u'].values, pesos_iob)[:len(df)]
    df['cob'] = np.convolve(df['carbs'].values, pesos_cob)[:len(df)]
    df['tendencia'] = df['glucosa'].diff()
    hora = df.index.hour + df.index.minute / 60
    df['hora_sin'] = np.sin(2 * np.pi * hora / 24); df['hora_cos'] = np.cos(2 * np.pi * hora / 24)
    return df

def a_local(ts): return ts.dt.tz_convert(ZONA_HORARIA).dt.tz_localize(None)

df_sgv = pd.read_csv(os.path.join(CARPETA_RES, 'nightscout_sgv_crudo.csv'))
df_trt = pd.read_csv(os.path.join(CARPETA_RES, 'nightscout_tratamientos_crudo.csv'))
df_sgv['ts'] = pd.to_datetime(df_sgv['ts'], utc=True, format='ISO8601')
s_g = pd.Series(df_sgv['glucosa'].astype(float).values, index=a_local(df_sgv['ts'])).sort_index()
s_g = s_g[(s_g >= 39) & (s_g <= 401)]
if len(df_trt):
    df_trt['ts'] = pd.to_datetime(df_trt['ts'], utc=True, format='ISO8601')
    tl = df_trt.assign(ts=a_local(df_trt['ts'])).set_index('ts').sort_index()
    s_b, s_c = tl['bolo_u'][tl['bolo_u'] > 0], tl['carbs'][tl['carbs'] > 0]
else:
    s_b = s_c = pd.Series(dtype=float)
df_propio = construir_base(s_g, s_b, s_c)

def ventanas(df, H):
    g = df['glucosa'].values
    F = np.stack([(df[c].fillna(0).values / ESCALAS[c]) for c in FEATURES], axis=1).astype('float32')
    n = len(df) - VENTANA - H + 1
    if n <= 0: return None
    valido = ~np.isnan(g); acum = np.concatenate([[0], np.cumsum(~valido)])
    ini = np.arange(n)
    ok = ((acum[ini + VENTANA] - acum[np.maximum(ini - 1, 0)]) == 0) & (ini >= 1) & valido[ini + VENTANA - 1 + H]
    ini = ini[ok]
    X = np.lib.stride_tricks.sliding_window_view(F, (VENTANA, F.shape[1]))[:, 0][ini]
    g_act, g_fut = g[ini + VENTANA - 1], g[ini + VENTANA - 1 + H]
    y = ((g_fut - g_act) / ESCALA_DELTA).astype('float32')
    return X.astype('float32'), y, g_act, g_fut, df.index[ini + VENTANA - 1 + H]

def dividir_propio(H):
    X, y, ga, gf, t = ventanas(df_propio, H); t = pd.DatetimeIndex(t)
    t1 = df_propio.index[int(len(df_propio) * SPLIT[0])]
    t2 = df_propio.index[int(len(df_propio) * (SPLIT[0] + SPLIT[1]))]
    m = pd.Timedelta(minutes=5 * (VENTANA + H))
    sel = {'train': t < t1, 'val': (t >= t1 + m) & (t < t2), 'test': t >= t2 + m}
    return {k: (X[s], y[s], ga[s], gf[s], t[s]) for k, s in sel.items()}

def predecir(m, X, g_act):
    return g_act + m.predict(X, batch_size=4096, verbose=0).ravel() * ESCALA_DELTA

v = df_propio['glucosa'].dropna()
print(f"Datos propios: {v.index.min():%d/%m/%Y} → {v.index.max():%d/%m/%Y} | {len(v)} lecturas")
print(f"Tiempo < 70: {(v < 70).mean()*100:.2f} % | < 54: {(v < 54).mean()*100:.2f} %")

## 3. Eventos de hipoglucemia y evaluación de alarmas

In [ ]:
def eventos_hipo(serie, umbral=HIPO, min_lect=MIN_LECTURAS_EVENTO, fusion_min=30):
    # Inicio de cada evento: ≥ min_lect lecturas seguidas < umbral. Eventos separados por < fusion_min se unen.
    bajo = (serie < umbral).values; idx = serie.index
    eventos, i, n = [], 0, len(bajo)
    while i < n:
        if bajo[i]:
            j = i
            while j + 1 < n and bajo[j + 1]: j += 1
            if j - i + 1 >= min_lect:
                if eventos and (idx[i] - eventos[-1][1]) < pd.Timedelta(minutes=fusion_min):
                    eventos[-1] = (eventos[-1][0], idx[j], min(eventos[-1][2], serie.iloc[i:j + 1].min()))
                else:
                    eventos.append((idx[i], idx[j], serie.iloc[i:j + 1].min()))
            i = j + 1
        else:
            i += 1
    return pd.DataFrame(eventos, columns=['inicio', 'fin', 'minimo'])

def evaluar_alarmas(t_pred, alarma, serie, eventos):
    # t_pred: instante en que se emite cada predicción; alarma: bool por instante; serie: glucosa real (grilla 5 min).
    t_pred = pd.DatetimeIndex(t_pred); alarma = np.asarray(alarma, bool)
    ta = t_pred[alarma]
    # episodios de alarma (agrupa alarmas separadas por < refractario)
    # Notificaciones: una alarma nueva notifica; si sigue activa, vuelve a notificar cada REFRACTARIO_MIN
    # (como las alertas repetidas de un CGM). Así una alarma encendida durante horas no cuenta como un solo aviso.
    inicios, ultimo_aviso = [], None
    for t in ta:
        if ultimo_aviso is None or (t - ultimo_aviso) >= pd.Timedelta(minutes=REFRACTARIO_MIN):
            inicios.append(t); ultimo_aviso = t
    inicios = pd.DatetimeIndex(inicios)
    # falsa alarma: no hay glucosa < 70 en los 60 min siguientes al inicio del aviso
    fa = 0
    for t in inicios:
        seg = serie[(serie.index >= t) & (serie.index <= t + pd.Timedelta(minutes=VENTANA_DETECCION_MIN))]
        if not (seg < HIPO).any(): fa += 1
    # Solo se evalúan eventos "anticipables": con al menos una predicción emitida en la hora previa.
    # Un evento que empieza justo después de un hueco del sensor no lo puede anticipar ningún método.
    w = pd.Timedelta(minutes=VENTANA_DETECCION_MIN)
    evaluables = [ini for ini in eventos['inicio'] if ((t_pred >= ini - w) & (t_pred <= ini)).any()]
    no_eval = len(eventos) - len(evaluables)
    # detección: alarma en [inicio - 60 min, inicio]; anticipación = inicio - primera alarma
    det, antic = [], []
    for ini in evaluables:
        cand = ta[(ta >= ini - pd.Timedelta(minutes=VENTANA_DETECCION_MIN)) & (ta <= ini)]
        det.append(len(cand) > 0)
        if len(cand): antic.append((ini - cand.min()).total_seconds() / 60)
    dias = max((t_pred.max() - t_pred.min()).total_seconds() / 86400, 1e-9)
    n_ev = len(evaluables)
    return {'eventos': n_ev, 'eventos_no_evaluables': no_eval, 'detectados': int(np.sum(det)),
            'sensibilidad': (np.mean(det) if n_ev else np.nan),
            'anticipacion_media_min': (np.mean(antic) if antic else np.nan),
            'detectados_>=15min': int(np.sum(np.array(antic) >= 15)) if antic else 0,
            'avisos': len(inicios), 'falsas_alarmas_dia': fa / dias, 'dias': dias,
            'tiempo_en_alarma_%': alarma.mean() * 100}

def ic_wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n; c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (c - h, c + h)

def eventos_en_periodo(t_pred, H):
    # eventos cuyo inicio cae dentro del período cubierto por las predicciones (con margen para poder anticiparlos)
    t = pd.DatetimeIndex(t_pred)
    s = df_propio['glucosa']
    s = s[(s.index >= t.min()) & (s.index <= t.max() + pd.Timedelta(minutes=5 * H))]
    ev = eventos_hipo(s)
    ev = ev[ev['inicio'] >= t.min() + pd.Timedelta(minutes=VENTANA_DETECCION_MIN)]
    return s, ev

## 4. Modelos: unificado (cargado) y versión con pérdida asimétrica
La pérdida asimétrica multiplica el error cuando la glucosa real está baja (< ~85 mg/dL) y, sobre todo, cuando el modelo **sobreestima** en esa zona. Ese es el error que deja pasar una hipo sin avisar. Se parte del modelo unificado ya entrenado y se reajusta solo con los datos propios de entrenamiento, usando la validación para la parada temprana.

In [ ]:
PESO_BAJO, PESO_SOBREESTIMA = 4.0, 12.0

def perdida_asimetrica(y_true, y_pred):
    delta, g_act = y_true[:, 0], y_true[:, 1]
    g_real = g_act + delta * ESCALA_DELTA
    g_pred = g_act + y_pred[:, 0] * ESCALA_DELTA
    bajo = tf.sigmoid((85.0 - g_real) / 6.0)
    sobre = tf.sigmoid((g_pred - g_real) / 3.0)
    w = 1.0 + PESO_BAJO * bajo + PESO_SOBREESTIMA * bajo * sobre
    return tf.reduce_mean(w * tf.square(y_pred[:, 0] - delta))

modelos, datos = {}, {}
for H in HORIZONTES:
    P = dividir_propio(H); datos[H] = P
    base = load_model(os.path.join(CARPETA_RES, f'modelo_lstm_dataset_unificado_{H*5}min.keras'))
    modelos[(H, 'unificado')] = base
    asim = clone_model(base); asim.set_weights(base.get_weights())
    asim.compile(optimizer=Adam(1e-4), loss=perdida_asimetrica)
    ruta_w = os.path.join(CARPETA_HIPO, f'modelo_lstm_unificado_asimetrico_{H*5}min.weights.h5')
    if os.path.exists(ruta_w):                      # ya entrenado en una corrida anterior: se reutiliza
        asim.load_weights(ruta_w); modelos[(H, 'asimetrico')] = asim
        print(f"H={H*5} min: modelo asimétrico cargado de la corrida anterior"); continue
    Xtr, ytr, gatr, _, _ = P['train']; Xva, yva, gava, _, _ = P['val']
    t0 = time.time()
    h = asim.fit(Xtr, np.stack([ytr, gatr], 1), validation_data=(Xva, np.stack([yva, gava], 1)),
                 epochs=60, batch_size=128, verbose=0,
                 callbacks=[EarlyStopping(patience=8, restore_best_weights=True), ReduceLROnPlateau(patience=4, factor=0.5, min_lr=1e-6)])
    print(f"H={H*5} min: modelo asimétrico {len(h.history['loss'])} épocas en {time.time()-t0:.0f}s")
    modelos[(H, 'asimetrico')] = asim
    asim.save_weights(ruta_w)

## 5. Estrategias de alarma
Cada estrategia tiene un parámetro (umbral θ o nivel α del intervalo). Se elige **en validación**: la menor tasa de falsas alarmas que alcance ≥ 90 % de sensibilidad; si ninguno lo alcanza, el de máxima sensibilidad. **El nivel α del límite conformal NO se ajusta con datos**: se fija a priori (0,05, 0,10 y 0,20, es decir, límites inferiores al 95 %, 90 % y 80 %), como se hace en predicción conformal. La validación propia tiene muy pocas hipos y ajustar α con ella lo vuelve inestable. La estrategia **híbrida** avisa si salta el límite conformal (α = 0,10) **o** la tendencia lineal con un umbral más estricto.

In [ ]:
def puntajes(P_split, H):
    X, _, g_act, _, t_obj = P_split
    gl = X[:, :, 0] * ESCALAS['glucosa']
    pendiente = (gl[:, -1] - gl[:, -4]) / 3.0           # mg/dL cada 5 min (últimos 15 min)
    s = {'Umbral actual (sin predicción)': g_act,
         'Tendencia lineal': g_act + pendiente * H,
         'LSTM unificado': predecir(modelos[(H, 'unificado')], X, g_act),
         'LSTM asimétrico': predecir(modelos[(H, 'asimetrico')], X, g_act)}
    return s, pd.DatetimeIndex(t_obj) - pd.Timedelta(minutes=5 * H)

GRILLA_UMBRAL = np.arange(50, 131, 2)
GRILLA_TEND_HIBRIDO = np.arange(40, 71, 2)

def q_conformal(res_val, alfa):
    n = len(res_val); return np.quantile(res_val, min(1.0, max(0.0, np.floor((n + 1) * alfa) / n)))

ALFAS_FIJOS = [0.05, 0.10, 0.20]   # nivel del límite inferior unilateral, fijado A PRIORI (no se ajusta con datos)
ALFA_HIBRIDO = 0.10

def alarma_de(nombre, par, s, res_val):
    # par: dict con los parámetros numéricos de la estrategia
    if nombre.startswith('LSTM + límite conformal'):
        return s['LSTM unificado'] + q_conformal(res_val, par['alfa']) < HIPO
    if nombre == 'Híbrido: conformal o tendencia':
        return (s['LSTM unificado'] + q_conformal(res_val, par['alfa']) < HIPO) | (s['Tendencia lineal'] < par['theta'])
    return s[nombre] < par['theta']

def candidatos(nombre):
    if nombre.startswith('LSTM + límite conformal'):
        return [{'alfa': float(nombre.split('= ')[1].rstrip(')'))}]          # fijo, sin ajuste
    if nombre == 'Híbrido: conformal o tendencia':
        return [{'alfa': ALFA_HIBRIDO, 'theta': t} for t in GRILLA_TEND_HIBRIDO]
    return [{'theta': u} for u in GRILLA_UMBRAL]

def texto(par): return ' · '.join(f"α = {v:.2f}" if k == 'alfa' else f"θ = {v:.0f} mg/dL" for k, v in par.items())

def elegir(filas):
    df = pd.DataFrame(filas)
    ok = df[df['sensibilidad'] >= SENSIBILIDAD_OBJETIVO]
    fila = (ok.sort_values(['falsas_alarmas_dia', 'tiempo_en_alarma_%']).iloc[0] if len(ok)
            else df.sort_values(['sensibilidad', 'falsas_alarmas_dia'], ascending=[False, True]).iloc[0])
    return fila['par']

ESTRATEGIAS = ['Umbral actual (sin predicción)', 'Tendencia lineal', 'LSTM unificado', 'LSTM asimétrico'] + \
              [f'LSTM + límite conformal (α = {a:.2f})' for a in ALFAS_FIJOS] + ['Híbrido: conformal o tendencia']
resultados, curvas, PARAMS, RES_VAL = [], {}, {}, {}
for H in HORIZONTES:
    P = datos[H]
    Xva, _, gava, gfva, _ = P['val']
    RES_VAL[H] = res_val = gfva - predecir(modelos[(H, 'unificado')], Xva, gava)
    sv, tv = puntajes(P['val'], H); serie_v, ev_v = eventos_en_periodo(tv, H)
    st, tt = puntajes(P['test'], H); serie_t, ev_t = eventos_en_periodo(tt, H)
    print(f"\nH={H*5} min | eventos de hipo en validación: {len(ev_v)} | en prueba: {len(ev_t)} "
          f"(los que siguen a un hueco del sensor se informan como 'no evaluables')")
    for nombre in ESTRATEGIAS:
        cands = candidatos(nombre)
        fv = [dict(evaluar_alarmas(tv, alarma_de(nombre, c, sv, res_val), serie_v, ev_v), par=c) for c in cands]
        ft = [dict(evaluar_alarmas(tt, alarma_de(nombre, c, st, res_val), serie_t, ev_t), par=c) for c in cands]
        curvas[(H, nombre)] = pd.DataFrame(ft)
        par = cands[0] if len(cands) == 1 else (elegir(fv) if len(ev_v) else cands[len(cands) // 2])
        PARAMS[(H, nombre)] = par
        r = next(f for f in ft if f['par'] == par)
        origen = ' (fijo a priori)' if len(cands) == 1 else ''
        resultados.append({'Horizonte_min': H * 5, 'Estrategia': nombre, 'Parámetro (elegido en val)': texto(par) + origen,
                           **{k: v for k, v in r.items() if k != 'par'}})

tabla = pd.DataFrame(resultados)
tabla['IC95 sensibilidad'] = [f"[{lo*100:.0f} %, {hi*100:.0f} %]" for lo, hi in (ic_wilson(int(d), int(e)) for d, e in zip(tabla['detectados'], tabla['eventos']))]
cols = ['Horizonte_min', 'Estrategia', 'Parámetro (elegido en val)', 'eventos', 'eventos_no_evaluables', 'detectados', 'sensibilidad', 'IC95 sensibilidad',
        'anticipacion_media_min', 'detectados_>=15min', 'falsas_alarmas_dia', 'tiempo_en_alarma_%']
tabla = tabla[cols]
tabla.to_csv(os.path.join(CARPETA_HIPO, 'tabla_deteccion_hipoglucemias.csv'), index=False)
pd.set_option('display.width', 220)
tabla.round(2)

## 6. Compromiso sensibilidad / falsas alarmas (conjunto de prueba)
Cada curva muestra qué sensibilidad se obtiene según cuántas falsas alarmas por día se toleren. Cuanto más arriba y a la izquierda, mejor.

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONTES), figsize=(7 * len(HORIZONTES), 5), squeeze=False)
colores = {'Umbral actual (sin predicción)': '#9e9e9e', 'Tendencia lineal': '#f9a825', 'LSTM unificado': '#1a73e8',
           'LSTM asimétrico': '#e53935', 'LSTM + límite conformal (α = 0.10)': '#43a047', 'Híbrido: conformal o tendencia': '#8e24aa'}
for ax, H in zip(axes[0], HORIZONTES):
    for nombre, c in colores.items():
        d = curvas[(H, nombre)].sort_values('falsas_alarmas_dia')
        ax.plot(d['falsas_alarmas_dia'], d['sensibilidad'] * 100, '-o', ms=3, color=c, label=nombre)
    ax.axhline(90, color='k', ls=':', lw=0.8); ax.set_xlim(0, 6)
    ax.set_xlabel('Falsas alarmas por día'); ax.set_ylabel('Sensibilidad (%)'); ax.set_title(f'Horizonte {H*5} min')
    ax.legend(fontsize=9, loc='lower right')
plt.tight_layout(); plt.savefig(os.path.join(CARPETA_HIPO, 'curva_sensibilidad_falsas_alarmas.png'), dpi=150); plt.show()

## 7. Costo en precisión general del modelo asimétrico
La pérdida asimétrica sacrifica algo de RMSE global para no pasar por alto las bajas. Esta celda cuantifica ese costo y el error en la zona baja.

In [ ]:
filas = []
for H in HORIZONTES:
    X, _, ga, gf, _ = datos[H]['test']
    for nombre in ('unificado', 'asimetrico'):
        p = predecir(modelos[(H, nombre)], X, ga); e = p - gf; bajo = gf < 80
        filas.append({'Horizonte_min': H * 5, 'Modelo': 'LSTM ' + nombre, 'RMSE global': np.sqrt(np.mean(e ** 2)),
                      'RMSE con glucosa real < 80': np.sqrt(np.mean(e[bajo] ** 2)) if bajo.any() else np.nan,
                      'Sesgo con glucosa real < 80 (+ = sobreestima)': e[bajo].mean() if bajo.any() else np.nan,
                      'n (< 80)': int(bajo.sum())})
tabla_costo = pd.DataFrame(filas)
tabla_costo.to_csv(os.path.join(CARPETA_HIPO, 'tabla_costo_asimetrico.csv'), index=False)
tabla_costo.round(2)

## 8. Validación externa con pacientes de prueba de OhioT1DM (opcional)
Con pocos eventos propios, los intervalos de la sensibilidad son anchos. Los pacientes de prueba de OhioT1DM **no se usaron para entrenar el modelo unificado** y tienen muchas más hipoglucemias, así que sirven para comprobar si las conclusiones se mantienen. Se usan los **mismos parámetros elegidos con la validación propia**.

In [ ]:
def leer_ohio(ruta):
    raiz = ET.parse(ruta).getroot(); fmt = '%d-%m-%Y %H:%M:%S'
    def serie(tag, cts, cval):
        el = raiz.find(tag)
        if el is None: return pd.Series(dtype=float)
        f = [(e.get(cts), e.get(cval)) for e in el.findall('event') if e.get(cval)]
        if not f: return pd.Series(dtype=float)
        return pd.Series([float(x[1]) for x in f], index=pd.to_datetime([x[0] for x in f], format=fmt)).sort_index()
    return raiz.get('id'), construir_base(serie('glucose_level', 'ts', 'value'), serie('bolus', 'ts_begin', 'dose'), serie('meal', 'ts', 'carbs'))

if USAR_OHIO:
    zip_drive = os.path.join(CARPETA_DRIVE, 'OhioT1DM.zip')
    if not os.path.exists('/content/OhioT1DM'):
        with zipfile.ZipFile(zip_drive) as z:
            z.extractall('/content/OhioT1DM', pwd=getpass.getpass('Contraseña OhioT1DM: ').encode())
    archivos = sorted(glob.glob('/content/OhioT1DM/**/test/*.xml', recursive=True))
    filas = []
    for H in HORIZONTES:
        tot = {}
        for f in archivos:
            pid, d = leer_ohio(f)
            v = ventanas(d, H)
            if v is None: continue
            X, _, ga, gf, t_obj = v
            t_em = pd.DatetimeIndex(t_obj) - pd.Timedelta(minutes=5 * H)
            serie = d['glucosa']; ev = eventos_hipo(serie)
            ev = ev[(ev['inicio'] >= t_em.min() + pd.Timedelta(minutes=VENTANA_DETECCION_MIN)) & (ev['inicio'] <= t_em.max())]
            s, _ = puntajes((X, None, ga, gf, t_obj), H)
            for nombre in ESTRATEGIAS:
                r = evaluar_alarmas(t_em, alarma_de(nombre, PARAMS[(H, nombre)], s, RES_VAL[H]), serie, ev)
                acc = tot.setdefault(nombre, {'eventos': 0, 'detectados': 0, 'fa': 0.0, 'dias': 0.0, 'antic': [], 'ta': []})
                acc['eventos'] += r['eventos']; acc['detectados'] += r['detectados']; acc['no_eval'] = acc.get('no_eval', 0) + r['eventos_no_evaluables']
                acc['fa'] += r['falsas_alarmas_dia'] * r['dias']; acc['dias'] += r['dias']; acc['ta'].append(r['tiempo_en_alarma_%'])
                if not np.isnan(r['anticipacion_media_min']): acc['antic'] += [r['anticipacion_media_min']] * r['detectados']
        for nombre, a in tot.items():
            lo, hi = ic_wilson(a['detectados'], a['eventos'])
            filas.append({'Horizonte_min': H * 5, 'Estrategia': nombre, 'Parámetro': texto(PARAMS[(H, nombre)]),
                          'eventos': a['eventos'], 'eventos_no_evaluables': a.get('no_eval', 0), 'sensibilidad': a['detectados'] / max(a['eventos'], 1),
                          'IC95 sensibilidad': f'[{lo*100:.0f} %, {hi*100:.0f} %]',
                          'anticipacion_media_min': np.mean(a['antic']) if a['antic'] else np.nan,
                          'falsas_alarmas_dia': a['fa'] / max(a['dias'], 1e-9),
                          'tiempo_en_alarma_%': np.mean(a['ta'])})
    tabla_ohio = pd.DataFrame(filas)
    tabla_ohio.to_csv(os.path.join(CARPETA_HIPO, 'tabla_deteccion_hipoglucemias_ohio.csv'), index=False)
    display(tabla_ohio.round(2))

## 9. Un ejemplo real: hipoglucemia del conjunto de prueba y cuándo avisó cada estrategia
Se muestra el evento más profundo entre los **anticipables** (con datos del sensor en la hora previa).

In [ ]:
H = HORIZONTES[0]
P = datos[H]; st, tt = puntajes(P['test'], H); serie_t, ev_t = eventos_en_periodo(tt, H)
w = pd.Timedelta(minutes=VENTANA_DETECCION_MIN)
ev_t = ev_t[[((tt >= i - w) & (tt <= i)).any() for i in ev_t['inicio']]]   # solo eventos anticipables
if len(ev_t):
    ev = ev_t.sort_values('minimo').iloc[0]
    ini, fin = ev['inicio'] - pd.Timedelta(hours=2), ev['fin'] + pd.Timedelta(hours=1)
    s = serie_t[(serie_t.index >= ini) & (serie_t.index <= fin)]
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(s.index, s.values, color='#1a73e8', lw=1.5, label='Glucosa real')
    ax.axhline(HIPO, color='red', ls='--', lw=0.8); ax.axvline(ev['inicio'], color='red', lw=0.8, alpha=0.5)
    for k, (nombre, c) in enumerate(colores.items()):
        al = alarma_de(nombre, PARAMS[(H, nombre)], st, RES_VAL[H])
        m = al & (tt >= ini) & (tt <= fin)
        ax.scatter(tt[m], np.full(m.sum(), 45 - 4 * k), marker='|', s=120, color=c, label=f'Aviso: {nombre}')
    ax.set_ylabel('mg/dL'); ax.set_title(f"Evento del {ev['inicio']:%d/%m %H:%M} (mínimo {ev['minimo']:.0f} mg/dL) — horizonte {H*5} min")
    ax.legend(fontsize=8, loc='upper right'); plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_HIPO, 'ejemplo_evento_hipo.png'), dpi=150); plt.show()
else:
    print('No hay eventos de hipoglucemia en el conjunto de prueba.')

## 10. Cómo leerlo (para el TFM)

- **Sensibilidad con anticipación** es la métrica principal de seguridad. Si el *Umbral actual* detecta casi todo pero con anticipación ~0 min, no previene nada. El valor de Glucast está en **avisar antes**.
- **Falsas alarmas/día**: la literatura sobre fatiga de alarmas sugiere mantenerlas bajas (del orden de 1–2 por día). Si una estrategia llega a 100 % de sensibilidad con 10 avisos diarios, en la práctica no sirve.
- **Pocos eventos → intervalos anchos.** Con un único paciente de lazo cerrado y TIR alto, hay pocas hipos en el período de prueba. Por eso se informa el IC 95 % (Wilson) y se valida en OhioT1DM.
- **Anti-sobreajuste.** Los umbrales se eligieron solo en validación y el modelo asimétrico se ajustó con parada temprana sobre validación. El conjunto de prueba no se usó para ninguna decisión.
- **Seguridad.** Glucast complementa las alarmas del Guardian 4 y de SmartGuard; **no las reemplaza** y no es un dispositivo médico certificado.